In [9]:
#biblioteke i učitavanje podataka

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import random
import warnings
warnings.filterwarnings('ignore')

#učitavanje podataka iz CSV fajla
df = pd.read_csv("iris.csv") 
X = df.drop('species', axis=1)
y = df['species']

#podjela podataka na trening i test skup (koristim ovo u oba zadatka)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Podaci su uspješno učitani iz iris.csv fajla i podijeljeni!")

Podaci su uspješno učitani iz iris.csv fajla i podijeljeni!


In [ ]:
#RandomSearch

#rječnik sa parametrima
OPCIJE = {
    "random_state": np.arange(0, 50).tolist(),
    "C": np.arange(0.1, 10.1, 0.1).tolist(),
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    # "break_ties": [True, False],  zbog ovog parametra je pucao kod na vježbama
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    "decision_function_shape": ['ovo', 'ovr'],
    "degree": [3, 4, 5],
    "max_iter": [-1, 100, 200],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5]
    # "verbose" smo izbacili iz opcija, jer ga sada postavljamo direktno u RandomizedSearchCV
}

print("Pokrećem RandomSearch...")

#inicijalizacija i pokretanje pretrage
random_search = RandomizedSearchCV(
    estimator=SVC(), 
    param_distributions=OPCIJE, 
    n_iter=100, 
    scoring='f1_weighted', 
    random_state=0,
    n_jobs=-1,   
    verbose=2       
)

#treniranje
random_search.fit(X, y)

#prikaz najboljih parametara i rezultata
print("\n=== Random Search rezultati ===")
print("Najbolji parametri:", random_search.best_params_)
print("Najbolji rezultat (F1 Score):", random_search.best_score_)

Pokrećem RandomSearch...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

=== Random Search rezultati ===
Najbolji parametri: {'tol': 0.0001, 'shrinking': True, 'random_state': 41, 'probability': True, 'max_iter': 100, 'kernel': 'rbf', 'gamma': 'auto', 'degree': 3, 'decision_function_shape': 'ovo', 'coef0': 0.0, 'class_weight': 'balanced', 'cache_size': 300, 'C': 1.5000000000000002}
Najbolji rezultat (F1 Score): 0.9866332497911445


In [ ]:
#Bat Algorithm (Aloritam šišmiša)

def objective(pozicija):
    c_val = max(0.01, pozicija[0])       
    gamma_val = max(0.0001, pozicija[1]) 
    
    model = SVC(C=c_val, kernel='rbf', gamma=gamma_val, random_state=0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    score = f1_score(y_test, y_pred, average='weighted')
    return 1.0 - score 

def popravi_granice(x, donja, gornja):
    x_popravljeno = np.copy(x)
    for j in range(len(x)):
        if x_popravljeno[j] < donja[j]:
            x_popravljeno[j] = donja[j]
        elif x_popravljeno[j] > gornja[j]:
            x_popravljeno[j] = gornja[j]
    return x_popravljeno

#parametri
N = 15                  
iteracije = 30          
dimenzije = 2           

donja_granica = np.array([0.1, 0.01])
gornja_granica = np.array([10.0, 1.0])

f_min = 0.0
f_max = 2.0
alpha = 0.9             
epsilon = 0.05          

#inicijalizacija
x_bat = np.zeros((N, dimenzije)) 
v = np.zeros((N, dimenzije))     
f = np.zeros(N)                  
loudness = np.ones(N) * 1.0      
pulse_rate = np.ones(N) * 0.5    
fitness = np.zeros(N)            

for i in range(N):
    x_bat[i] = donja_granica + np.random.rand(dimenzije) * (gornja_granica - donja_granica)
    fitness[i] = objective(x_bat[i])

best_idx = np.argmin(fitness)
best = np.copy(x_bat[best_idx])
best_fitness = fitness[best_idx]

print(f"Početni najbolji F1 Score: {1.0 - best_fitness:.4f}")
print("Započinje Bat Algorithm...\n")

#glavna petlja
for t in range(iteracije):
    average_loudness = np.mean(loudness)
    
    for i in range(N): 
        beta = random.uniform(0, 1) 
        f[i] = f_min + (f_max - f_min) * beta
        v[i] = v[i] + (x_bat[i] - best) * f[i]
        x_new = x_bat[i] + v[i]
        
        if random.uniform(0, 1) > pulse_rate[i]:
            x_new = best + epsilon * average_loudness * np.random.randn(dimenzije)
            
        x_new = popravi_granice(x_new, donja_granica, gornja_granica)
        fitness_new = objective(x_new)
        
        if fitness_new < fitness[i] and random.uniform(0, 1) < loudness[i]:
            x_bat[i] = np.copy(x_new)         
            fitness[i] = fitness_new          
            loudness[i] = alpha * loudness[i] 
            
        if fitness_new < best_fitness:
            best = np.copy(x_new)
            best_fitness = fitness_new
            

    print(f"Iteracija {t+1:2d}/{iteracije} | Trenutno najbolji F1: {1.0 - best_fitness:.4f} | C={best[0]:.4f}, gamma={best[1]:.4f}")

print("\n=== Bat Algorithm završen ===")
print(f"Najbolji F1 Score pronađen: {1.0 - best_fitness:.4f}")
print(f"Pronađeni hiperparametri -> C: {best[0]:.4f}, gamma: {best[1]:.4f}")

Početni najbolji F1 Score: 1.0000
Započinje Bat Algorithm...

Iteracija  1/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  2/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  3/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  4/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  5/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  6/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  7/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  8/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija  9/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija 10/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija 11/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija 12/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
Iteracija 13/30 | Trenutno najbolji F1: 1.0000 | C=8.2791, gamma=0.1230
It